# HW2 Part 2 - RAG pipeline over movie Wikipedia pages

This notebook builds a small retrieval-augmented generation (RAG) pipeline using LangChain,
over the Wikipedia pages for 10 well-known movies.

Here's what it does:

- loads the 10 Wikipedia pages using a LangChain document loader
- splits them into chunks (500 characters, 50 overlap)
- embeds the chunks and stores them in a vector store (Chroma)
- wires up the pipeline by hand - retriever, prompt template, LLM - instead of using one of
  LangChain's prebuilt chain classes, so every step is visible
- asks it 5 questions that are all answerable from the pages, and shows both the retrieved
  chunks and the generated answer for each
- reruns 2 of those questions with a different chunk size/overlap and compares
- manually checks, for each question, whether the right answer was actually retrieved in the
  top 3 chunks, and works out a retrieval success rate
- digs into at least 2 concrete cases where the pipeline messed up, and explains why

Everything here runs locally with free HuggingFace models - no API key needed. Embeddings come
from `sentence-transformers/all-MiniLM-L6-v2` (small, 384-dim, runs fine on CPU) and the
generator is `google/flan-t5-base` (~250M params, also CPU-friendly). Both models get
downloaded once (about 1GB total) and then everything runs offline.


## 1. Setup

In [1]:
# (pip install cell -- skipped for local execution; dependencies are already installed)
# !pip -q install -U langchain langchain-community langchain-huggingface langchain-chroma     wikipedia chromadb sentence-transformers "transformers==4.44.2" accelerate sentencepiece --upgrade
# print("If Colab asks you to restart the runtime, restart once and re-run from this cell.")

In [2]:
import os, time, textwrap
import pandas as pd

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

print("Running fully locally with free HuggingFace models -- no API key required.")

Running fully locally with free HuggingFace models -- no API key required.


## Loading the 10 movie pages

`WikipediaLoader` is LangChain's document loader for pulling pages straight from Wikipedia's
API. Each page comes back as a `Document` with the text in `page_content` and some metadata
attached.


In [3]:
import wikipedia.wikipedia as _wikipedia_lib
_wikipedia_lib.API_URL = "https://en.wikipedia.org/w/api.php"  # the package defaults to http://, which Wikipedia now blocks with 403

from langchain_community.document_loaders import WikipediaLoader

MOVIES = [
    "The Shawshank Redemption",
    "Inception (film)",
    "The Godfather",
    "Pulp Fiction",
    "The Dark Knight",
    "Forrest Gump",
    "The Matrix",
    "Interstellar (film)",
    "Parasite (2019 film)",
    "Titanic (1997 film)",
]

def load_with_retry(title, attempts=6, base_delay=5.0):
    """Wikipedia occasionally rate-limits/returns an empty response; retry with a
    growing backoff instead of failing the whole run on a transient hiccup."""
    last_err = None
    for attempt in range(1, attempts + 1):
        try:
            return WikipediaLoader(query=title, load_max_docs=1, doc_content_chars_max=20000).load()
        except Exception as e:
            last_err = e
            print(f"   retry {attempt}/{attempts-1} for {title!r} after {type(e).__name__}: {e}")
            time.sleep(base_delay * attempt)
    print(f"!! giving up on {title!r} after {attempts} attempts ({type(last_err).__name__}); skipping")
    return []

docs = []
for title in MOVIES:
    loaded = load_with_retry(title)
    if not loaded:
        print(f"!! nothing found for {title}")
        continue
    d = loaded[0]
    d.metadata["movie"] = title
    docs.append(d)
    time.sleep(2.0)   # be polite to the Wikipedia API and avoid transient rate-limit errors

print(f"loaded {len(docs)} documents")
for d in docs:
    print(f" - {d.metadata.get('title', d.metadata['movie']):<35s} {len(d.page_content):>6,} chars")

C:\Users\Admin\AppData\Local\Temp\ipykernel_20276\2562578836.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


loaded 10 documents
 - The Shawshank Redemption            20,000 chars
 - Inception                           20,000 chars
 - The Godfather                       20,000 chars
 - Pulp Fiction                        20,000 chars
 - The Dark Knight                     20,000 chars
 - Forrest Gump                        20,000 chars
 - The Matrix                          20,000 chars
 - Interstellar (film)                 20,000 chars
 - Parasite (2019 film)                20,000 chars
 - Titanic (1997 film)                 20,000 chars


In [4]:
docs[0].metadata

{'title': 'The Shawshank Redemption',
 'summary': 'The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 1982 Stephen King novella Rita Hayworth and Shawshank Redemption. The film tells the story of banker Andy Dufresne (Tim Robbins), who is sentenced to life in Shawshank State Penitentiary for the murders of his wife and her lover, despite his claims of innocence. Over the following two decades, he befriends a fellow prisoner, contraband smuggler Ellis "Red" Redding (Morgan Freeman), and becomes instrumental in a money laundering operation led by the prison warden Samuel Norton (Bob Gunton). William Sadler, Clancy Brown, Gil Bellows, and James Whitmore appear in supporting roles.\nDarabont purchased the film rights to King\'s story in 1987, but development did not begin until five years later, when he wrote the script over eight weeks. Two weeks after submitting his script to Castle Rock Entertainment, Darabont secured a $25 millio

In [5]:
print(docs[0].page_content[:800])

The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 1982 Stephen King novella Rita Hayworth and Shawshank Redemption. The film tells the story of banker Andy Dufresne (Tim Robbins), who is sentenced to life in Shawshank State Penitentiary for the murders of his wife and her lover, despite his claims of innocence. Over the following two decades, he befriends a fellow prisoner, contraband smuggler Ellis "Red" Redding (Morgan Freeman), and becomes instrumental in a money laundering operation led by the prison warden Samuel Norton (Bob Gunton). William Sadler, Clancy Brown, Gil Bellows, and James Whitmore appear in supporting roles.
Darabont purchased the film rights to King's story in 1987, but development did not begin until five years 


## Splitting into chunks

We're using `RecursiveCharacterTextSplitter`, which tries to break text on paragraph breaks
first, then sentences, then words - only falling back to a hard cut in the middle of a word if
nothing smaller fits. That keeps chunks reading like actual sentences instead of getting cut
off arbitrarily.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE, CHUNK_OVERLAP = 500, 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks "
      f"(chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

from collections import Counter
per_movie = Counter(c.metadata["movie"] for c in chunks)
pd.Series(per_movie, name="n_chunks").sort_index()

10 documents -> 601 chunks (chunk_size=500, overlap=50)


Forrest Gump                60
Inception (film)            62
Interstellar (film)         62
Parasite (2019 film)        59
Pulp Fiction                58
The Dark Knight             62
The Godfather               60
The Matrix                  61
The Shawshank Redemption    60
Titanic (1997 film)         57
Name: n_chunks, dtype: int64

In [7]:
print(chunks[5].metadata)
print("-"*60)
print(chunks[5].page_content)

{'title': 'The Shawshank Redemption', 'summary': 'The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 1982 Stephen King novella Rita Hayworth and Shawshank Redemption. The film tells the story of banker Andy Dufresne (Tim Robbins), who is sentenced to life in Shawshank State Penitentiary for the murders of his wife and her lover, despite his claims of innocence. Over the following two decades, he befriends a fellow prisoner, contraband smuggler Ellis "Red" Redding (Morgan Freeman), and becomes instrumental in a money laundering operation led by the prison warden Samuel Norton (Bob Gunton). William Sadler, Clancy Brown, Gil Bellows, and James Whitmore appear in supporting roles.\nDarabont purchased the film rights to King\'s story in 1987, but development did not begin until five years later, when he wrote the script over eight weeks. Two weeks after submitting his script to Castle Rock Entertainment, Darabont secured a $25 million

## Embeddings and vector store

Each chunk gets embedded with a small local model (`all-MiniLM-L6-v2`, about 80MB, runs fine
on CPU) and the vectors go into a Chroma collection - basically an in-process vector database
that the retriever can query by similarity later. No API calls involved beyond the initial
model download.


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="movie_wiki_v1",
)
print("vector store built:", vectorstore._collection.count(), "vectors")

C:\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


vector store built: 601 vectors


## The local LLM

For the generator we're using `google/flan-t5-base`, a small local sequence-to-sequence model
wrapped as a LangChain `HuggingFacePipeline`. It's standing in for something like `gpt-4o-mini`
- everything else in the pipeline (the prompt template, the retriever, how it's all wired
together) doesn't care which LLM is plugged in.


In [9]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, T5ForConditionalGeneration

# Load model/tokenizer explicitly (rather than pipeline(model=<name>)) so transformers
# doesn't try to also probe for a TensorFlow checkpoint, which fails on machines with a
# Keras 3 install (unrelated RuntimeError) even when only PyTorch is actually needed.
_t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
_t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
gen_pipe = pipeline("text2text-generation", model=_t5_model, tokenizer=_t5_tok, max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=gen_pipe)
print("Local LLM ready: google/flan-t5-base (CPU)")

C:\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Local LLM ready: google/flan-t5-base (CPU)


## Wiring the pipeline together

Instead of calling something like `RetrievalQA.from_chain_type(...)` and letting LangChain
hide the details, we build the pipeline out of its actual pieces so it's clear what's
happening at each step:

retriever (pulls chunks from the vector store) -> prompt template (formats those chunks plus
the question) -> LLM (writes the answer) -> output parser (just cleans up the string).


In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a helpful assistant answering questions about movies using only the "
        "context below, which was retrieved from Wikipedia.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Instructions: Answer using ONLY the context above. If the answer is not contained "
        "in the context, say \"I don't know based on the provided context.\" "
        "Be concise (1-3 sentences).\n\nAnswer:"
    ),
)

def format_docs(retrieved_docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('movie','?')}] {d.page_content}" for d in retrieved_docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

print("pipeline ready: retriever -> PromptTemplate -> LLM -> StrOutputParser")

pipeline ready: retriever -> PromptTemplate -> LLM -> StrOutputParser


## Asking 5 questions

Each of these questions should be answerable from the pages we loaded. For each one we print
the top-3 retrieved chunks (which movie they came from, and a preview of the text) along with
the final answer the model generated.


In [11]:
QUESTIONS = [
    "What crime is Andy Dufresne convicted of in The Shawshank Redemption?",
    "What is the name of the spinning top object used in Inception?",
    "Who directed The Godfather?",
    "In Pulp Fiction, what do Vincent and Jules do for a living?",
    "Who plays the Joker in The Dark Knight?",
]

def run_question(question, retriever, chain, k_show=3):
    retrieved = retriever.invoke(question)[:k_show]
    answer = chain.invoke(question)
    return retrieved, answer

def show_result(question, retrieved, answer):
    print("="*100)
    print("Q:", question)
    print("-"*100)
    for i, d in enumerate(retrieved, start=1):
        snippet = d.page_content.replace("\n", " ")
        print(f"  [{i}] ({d.metadata.get('movie')})  {snippet[:220]}...")
    print("-"*100)
    print("ANSWER:", textwrap.fill(answer, 96))
    print()

results_v1 = []
for q in QUESTIONS:
    retrieved, answer = run_question(q, retriever, rag_chain)
    show_result(q, retrieved, answer)
    results_v1.append({"question": q, "retrieved": retrieved, "answer": answer})

Q: What crime is Andy Dufresne convicted of in The Shawshank Redemption?
----------------------------------------------------------------------------------------------------
  [1] (The Shawshank Redemption)  == Plot == In 1947, Portland, Maine, banker Andy Dufresne arrives at Shawshank State Prison to serve two consecutive life sentences for murdering his wife and her lover. He is befriended by Ellis Boyd "Red" Redding, a co...
  [2] (The Shawshank Redemption)  . While guards search for him, Andy poses as Randall Stephens and withdraws over $370,000 of the laundered money from various banks, before mailing the ledger to a local newspaper. State police arrive at Shawshank and ta...
  [3] (The Shawshank Redemption)  . Just as Andy can be interpreted as a Christ-like figure, he can be seen as a Zarathustra-like prophet offering escape through education and the experience of freedom. Film critic Roger Ebert argued that The Shawshank R...
----------------------------------------------------

Q: What is the name of the spinning top object used in Inception?
----------------------------------------------------------------------------------------------------
  [1] (Inception (film))  Inception is a 2010  science fiction heist film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professional thief who steals information b...
  [2] (Inception (film))  . Cobb uses a top that spins indefinitely in a dream to verify that he is in the real world, but joins his children before he can observe the result....
  [3] (Inception (film))  . Nolan took a long time re-writing the script in order "to make sure that the emotional journey of his [DiCaprio's] character was the driving force of the movie." On February 11, 2009, it was announced that Warner Bros....
----------------------------------------------------------------------------------------------------
ANSWER: Cobb



Q: Who directed The Godfather?
----------------------------------------------------------------------------------------------------
  [1] (The Godfather)  The Godfather is a 1972 American epic gangster film directed by Francis Ford Coppola, who co-wrote the screenplay with Mario Puzo based on Puzo's best-selling 1969 novel. The film features an ensemble cast that includes ...
  [2] (The Godfather)  . It was followed by the sequels The Godfather Part II (1974) and The Godfather Part III (1990)....
  [3] (The Godfather)  . It is the first installment in The Godfather trilogy, which chronicles the Corleone family under patriarch Vito Corleone (Brando) and the transformation of his youngest son, Michael Corleone (Pacino), from reluctant fa...
----------------------------------------------------------------------------------------------------
ANSWER: Francis Ford Coppola



Q: In Pulp Fiction, what do Vincent and Jules do for a living?
----------------------------------------------------------------------------------------------------
  [1] (Pulp Fiction)  Jules and Vincent meet Marsellus at a bar, where Marsellus is bribing an aging Butch to intentionally lose in his upcoming boxing match. The following night, Vincent purchases heroin from his drug dealer, Lance. He shoot...
  [2] (Pulp Fiction)  . Jules tells Vincent that he plans to retire from his life of crime, convinced that their survival at the apartment was divine intervention....
  [3] (Pulp Fiction)  While Jules is driving, Vincent accidentally shoots Marvin in the head, covering them in blood. They hide the car at the home of Jules' friend Jimmie, who demands they dispose of Marvin's corpse and the blood-stained car...
----------------------------------------------------------------------------------------------------
ANSWER: crime



Q: Who plays the Joker in The Dark Knight?
----------------------------------------------------------------------------------------------------
  [1] (The Dark Knight)  . Their efforts are derailed by the Joker, an anarchistic mastermind who seeks to test how far Batman will go to save the city from chaos. The cast includes Christian Bale, Michael Caine, Heath Ledger, Gary Oldman, Aaron...
  [2] (The Dark Knight)  . Melinda McGraw, Nathan Gamble, and Hannah Gunn portray Gordon's wife Barbara, his son James Jr., and his daughter, respectively. The Dark Knight features several cameo appearances from Cillian Murphy, who reprises his ...
  [3] (The Dark Knight)  The Dark Knight was marketed with an innovative interactive viral campaign that initially focused on countering criticism of Ledger's casting by those who believed he was a poor choice to portray the Joker. Ledger died f...
----------------------------------------------------------------------------------------------------
ANSWER: 

In [12]:
rows = []
for r in results_v1:
    for i, d in enumerate(r["retrieved"], start=1):
        rows.append({
            "question": r["question"][:60] + "...",
            "rank": i,
            "movie": d.metadata.get("movie"),
            "chunk_preview": d.page_content[:150].replace("\n", " ") + "...",
        })
chunks_v1_df = pd.DataFrame(rows)
chunks_v1_df.to_csv("results/retrieved_chunks_v1.csv", index=False)
chunks_v1_df

,question,rank,movie,chunk_preview
0,What crime is Andy Dufresne convicted of in Th...,1,The Shawshank Redemption,"== Plot == In 1947, Portland, Maine, banker An..."
1,What crime is Andy Dufresne convicted of in Th...,2,The Shawshank Redemption,". While guards search for him, Andy poses as R..."
2,What crime is Andy Dufresne convicted of in Th...,3,The Shawshank Redemption,. Just as Andy can be interpreted as a Christ-...
3,What is the name of the spinning top object us...,1,Inception (film),Inception is a 2010 science fiction heist fil...
4,What is the name of the spinning top object us...,2,Inception (film),. Cobb uses a top that spins indefinitely in a...
5,What is the name of the spinning top object us...,3,Inception (film),. Nolan took a long time re-writing the script...
6,Who directed The Godfather?...,1,The Godfather,The Godfather is a 1972 American epic gangster...
7,Who directed The Godfather?...,2,The Godfather,. It was followed by the sequels The Godfather...
8,Who directed The Godfather?...,3,The Godfather,. It is the first installment in The Godfather...
9,"In Pulp Fiction, what do Vincent and Jules do ...",1,Pulp Fiction,"Jules and Vincent meet Marsellus at a bar, whe..."


In [13]:
answers_v1_df = pd.DataFrame([{"question": r["question"], "answer": r["answer"]} for r in results_v1])
answers_v1_df.to_csv("results/answers_v1.csv", index=False)
answers_v1_df

,question,answer
0,What crime is Andy Dufresne convicted of in Th...,murdering his wife and her lover
1,What is the name of the spinning top object us...,Cobb
2,Who directed The Godfather?,Francis Ford Coppola
3,"In Pulp Fiction, what do Vincent and Jules do ...",crime
4,Who plays the Joker in The Dark Knight?,Ledger


## Rerunning 2 questions with a different chunk size

Here we rebuild the splitter and vector store with bigger chunks - 1000 characters, 100
overlap, instead of 500/50 - and rerun the first two questions to see what changes.


In [14]:
CHUNK_SIZE_2, CHUNK_OVERLAP_2 = 1000, 100

splitter_2 = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_2, chunk_overlap=CHUNK_OVERLAP_2,
    length_function=len, separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_2 = splitter_2.split_documents(docs)
print(f"{len(docs)} documents -> {len(chunks_2)} chunks "
      f"(chunk_size={CHUNK_SIZE_2}, overlap={CHUNK_OVERLAP_2})  "
      f"[was {len(chunks)} chunks at 500/50]")

vectorstore_2 = Chroma.from_documents(
    documents=chunks_2, embedding=embeddings, collection_name="movie_wiki_v2",
)
retriever_2 = vectorstore_2.as_retriever(search_type="similarity", search_kwargs={"k": 3})

rag_chain_2 = (
    {"context": retriever_2 | format_docs, "question": RunnablePassthrough()}
    | PROMPT | llm | StrOutputParser()
)

10 documents -> 330 chunks (chunk_size=1000, overlap=100)  [was 601 chunks at 500/50]


In [15]:
RERUN_QUESTIONS = QUESTIONS[:2]   # questions 1 and 2

results_v2 = []
for q in RERUN_QUESTIONS:
    retrieved, answer = run_question(q, retriever_2, rag_chain_2)
    show_result(q, retrieved, answer)
    results_v2.append({"question": q, "retrieved": retrieved, "answer": answer})

Token indices sequence length is longer than the specified maximum sequence length for this model (604 > 512). Running this sequence through the model will result in indexing errors


Q: What crime is Andy Dufresne convicted of in The Shawshank Redemption?
----------------------------------------------------------------------------------------------------
  [1] (The Shawshank Redemption)  == Plot == In 1947, Portland, Maine, banker Andy Dufresne arrives at Shawshank State Prison to serve two consecutive life sentences for murdering his wife and her lover. He is befriended by Ellis Boyd "Red" Redding, a co...
  [2] (The Shawshank Redemption)  At the next day's roll call, the guards find Andy's cell empty. An irate Norton throws a stone at a poster of Raquel Welch hanging on the cell wall, revealing a tunnel that Andy had dug with his rock hammer over 19 years...
  [3] (The Shawshank Redemption)  While some Christian viewers interpret Zihuatanejo as heaven, film critic Mark Kermode wrote that it can also be interpreted as a Nietzschean form of guiltlessness achieved outside traditional notions of good and evil, w...
----------------------------------------------------

Q: What is the name of the spinning top object used in Inception?
----------------------------------------------------------------------------------------------------
  [1] (Inception (film))  Inception is a 2010  science fiction heist film written and directed by Christopher Nolan, who also produced it with his wife Emma Thomas. The film stars Leonardo DiCaprio as a professional thief who steals information b...
  [2] (Inception (film))  Inception's premiere was held in London on July 8, 2010; it was released in both conventional and IMAX theaters beginning on July 16, 2010. Inception grossed $839 million worldwide, becoming the fourth-highest-grossing f...
  [3] (Inception (film))  Nolan had been trying to work with Leonardo DiCaprio for years and met him several times, but was unable to recruit him for any of his films until Inception. DiCaprio finally agreed because he was "intrigued by this conc...
-----------------------------------------------------------------------------------

### Comparing 500/50 against 1000/100

In [16]:
compare_rows = []
for q in RERUN_QUESTIONS:
    v1 = next(r for r in results_v1 if r["question"] == q)
    v2 = next(r for r in results_v2 if r["question"] == q)
    compare_rows.append({
        "question": q,
        "answer (500/50)":  v1["answer"],
        "answer (1000/100)": v2["answer"],
        "top1 movie (500/50)":  v1["retrieved"][0].metadata.get("movie") if v1["retrieved"] else None,
        "top1 movie (1000/100)": v2["retrieved"][0].metadata.get("movie") if v2["retrieved"] else None,
        "n chars top1 (500/50)":  len(v1["retrieved"][0].page_content) if v1["retrieved"] else 0,
        "n chars top1 (1000/100)": len(v2["retrieved"][0].page_content) if v2["retrieved"] else 0,
    })
compare_df = pd.DataFrame(compare_rows)
compare_df.to_csv("results/chunking_comparison.csv", index=False)
compare_df

,question,answer (500/50),answer (1000/100),top1 movie (500/50),top1 movie (1000/100),n chars top1 (500/50),n chars top1 (1000/100)
0,What crime is Andy Dufresne convicted of in Th...,murdering his wife and her lover,murdering his wife and her lover,The Shawshank Redemption,The Shawshank Redemption,443,443
1,What is the name of the spinning top object us...,Cobb,I don't know,Inception (film),Inception (film),383,550


Bigger chunks didn't change the first question at all - Shawshank's answer stayed correct
either way, since the whole fact ("convicted of murdering his wife and her lover") sits well
inside a single 500-character chunk regardless. The Inception question is more interesting:
at 500/50, the model got as far as answering "Cobb" (the character's name, not the object),
and at 1000/100 it actually got worse and answered "I don't know" - the bigger chunk pulled in
a different section of the page entirely (box office numbers) and pushed the sentence about
the spinning top out of the top-1 slot. So bigger chunks aren't a free win: they can carry more
context per chunk, but they also change which chunk wins the similarity search, and that can
go either way.


## Checking retrieval quality by hand

For each of the 5 original questions, we go find the actual passage in the source page that
answers it, then check whether that information showed up anywhere in the top-3 retrieved
chunks - and if so, at what rank.


In [17]:
# Ground truth: the source passage that answers each question (fill in from the docs above)
GROUND_TRUTH = {
    QUESTIONS[0]: {
        "source_passage": "Andy Dufresne, a banker, is convicted of murdering his wife and her lover.",
        "movie": "The Shawshank Redemption",
    },
    QUESTIONS[1]: {
        "source_passage": "Cobb carries a spinning top, his totem, to test whether he is dreaming.",
        "movie": "Inception (film)",
    },
    QUESTIONS[2]: {
        "source_passage": "The Godfather was directed by Francis Ford Coppola.",
        "movie": "The Godfather",
    },
    QUESTIONS[3]: {
        "source_passage": "Vincent Vega and Jules Winnfield are hitmen working for crime boss Marsellus Wallace.",
        "movie": "Pulp Fiction",
    },
    QUESTIONS[4]: {
        "source_passage": "Heath Ledger plays the Joker in The Dark Knight.",
        "movie": "The Dark Knight",
    },
}

def first_relevant_rank(retrieved, movie, keyword_hint=None):
    """Return the 1-indexed rank of the first chunk from the right movie
    (optionally also containing a keyword hint) or None if none of the top-k qualify."""
    for i, d in enumerate(retrieved, start=1):
        if d.metadata.get("movie") == movie:
            if keyword_hint is None or keyword_hint.lower() in d.page_content.lower():
                return i
    return None

# NOTE: inspect the printed retrieved chunks in section 5 and set these hints/ranks manually --
# this cell provides a *keyword-based* automatic first pass, which you should sanity check
# against the actual printed chunk text.
HINTS = {
    QUESTIONS[0]: "murder",
    QUESTIONS[1]: "top",
    QUESTIONS[2]: "coppola",
    QUESTIONS[3]: "hitmen",
    QUESTIONS[4]: "ledger",
}

eval_rows = []
for r in results_v1:
    q = r["question"]
    gt = GROUND_TRUTH[q]
    rank = first_relevant_rank(r["retrieved"], gt["movie"], HINTS[q])
    eval_rows.append({
        "question": q,
        "ground_truth_movie": gt["movie"],
        "ground_truth_passage": gt["source_passage"],
        "top3_contains_answer": "Yes" if rank is not None else "No",
        "rank_of_first_relevant_chunk": rank if rank is not None else "-",
    })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv("results/retrieval_evaluation.csv", index=False)
eval_df

,question,ground_truth_movie,ground_truth_passage,top3_contains_answer,rank_of_first_relevant_chunk
0,What crime is Andy Dufresne convicted of in Th...,The Shawshank Redemption,"Andy Dufresne, a banker, is convicted of murde...",Yes,1
1,What is the name of the spinning top object us...,Inception (film),"Cobb carries a spinning top, his totem, to tes...",Yes,1
2,Who directed The Godfather?,The Godfather,The Godfather was directed by Francis Ford Cop...,Yes,1
3,"In Pulp Fiction, what do Vincent and Jules do ...",Pulp Fiction,Vincent Vega and Jules Winnfield are hitmen wo...,No,-
4,Who plays the Joker in The Dark Knight?,The Dark Knight,Heath Ledger plays the Joker in The Dark Knight.,Yes,1


In [18]:
success = (eval_df["top3_contains_answer"] == "Yes").sum()
total = len(eval_df)
retrieval_success_rate = success / total
print(f"Retrieval Success Rate = {success}/{total} = {retrieval_success_rate:.0%}")

Retrieval Success Rate = 4/5 = 80%


One thing to flag: the Yes/No check above uses a simple keyword match (does the retrieved
chunk contain a hint word like "murder" or "coppola") as a quick first pass. It's not perfect -
a chunk could contain the hint word in an unrelated sentence, or the real answer could be
phrased without that exact word - so the actual retrieved chunks were read by hand against the
ground-truth passage before trusting the Yes/No column.


## Where things went wrong

Two real problems, based on what the pipeline actually retrieved and answered:


In [19]:
failures = []

# --- Actual failures observed in this run (from the printed chunks/answers in section 5/6) ------
failures.append({
    "failure_id": 1,
    "question": QUESTIONS[1],   # the Inception "totem" question
    "category": "chunk boundary separated important information",
    "evidence": (
        "At 500/50 the top-1 chunk is a generic film-intro paragraph and the top-2 chunk "
        "separately states 'Cobb uses a top that spins indefinitely in a dream to verify that "
        "he is in the real world' -- the LLM answered just 'Cobb' (the character, not the "
        "object) instead of 'a spinning top / his totem'. At 1000/100 the retrieved top-1 chunk "
        "shifted to a different section entirely (box-office gross) and the model answered "
        "'I don't know' -- the larger chunks pushed the actually-relevant sentence out of the "
        "top-1 slot, trading one failure mode for another rather than fixing it."
    ),
    "impact": "Neither chunk size gets a fully correct, well-grounded answer to this question; "
              "the underlying fact ('spinning top'/'totem') is present in the corpus but never "
              "lands in the single top-ranked chunk the model leans on most.",
})

failures.append({
    "failure_id": 2,
    "question": QUESTIONS[3],   # the Pulp Fiction "job" question
    "category": "relevant chunk ranked too low / correct document retrieved but key fact missing from top-3",
    "evidence": (
        "All 3 retrieved chunks are correctly from Pulp Fiction's Plot section (movie name "
        "matches), so retrieval found the right *document*, but none of the top-3 chunks state "
        "the specific job title -- rank 1 shows Jules/Vincent at a bar with Marsellus, rank 2 "
        "mentions Jules' 'life of crime', rank 3 shows an unrelated shooting scene. The exact "
        "sentence identifying them as hitmen/hired killers for Marsellus Wallace apparently sits "
        "in a different 500-char chunk that did not make the top-3 similarity cut. The LLM, given "
        "only 'life of crime' as its best clue, generated the vague answer 'crime' instead of the "
        "specific job title."
    ),
    "impact": "Retrieval Success Rate counts this question as a miss (rank_of_first_relevant_chunk "
              "= '-') even though the correct movie was found, illustrating that 'right document' "
              "and 'right chunk containing the specific fact' are different bars -- vector "
              "similarity over short chunks can favor generic contextual overlap over the one "
              "sentence that actually answers the question.",
})
# --------------------------------------------------------------------------------------------

failures_df = pd.DataFrame(failures)
failures_df.to_csv("results/rag_failures.csv", index=False)
failures_df

,failure_id,question,category,evidence,impact
0,1,What is the name of the spinning top object us...,chunk boundary separated important information,At 500/50 the top-1 chunk is a generic film-in...,"Neither chunk size gets a fully correct, well-..."
1,2,"In Pulp Fiction, what do Vincent and Jules do ...",relevant chunk ranked too low / correct docume...,All 3 retrieved chunks are correctly from Pulp...,Retrieval Success Rate counts this question as...


Both of these came straight from the printed chunks and answers earlier in the notebook -
nothing here is hypothetical. A few other kinds of RAG failures worth knowing about, in case a
different run turns up different problems: retrieval missing the right document entirely,
a relevant chunk technically being retrieved but ranked too low to matter, the LLM
hallucinating a fact that isn't in any retrieved chunk, or ambiguous movie titles pulling in
chunks from the wrong film. This run's two failures happen to be a chunk-boundary problem and
a "right document, wrong chunk" problem, but the underlying pipeline could produce any of the
others depending on the questions and documents.


## Wrapping up

In [20]:
print("="*70)
print("PART 2 SUMMARY")
print("="*70)
print(f"Documents loaded         : {len(docs)}")
print(f"Chunks (500/50)          : {len(chunks)}")
print(f"Chunks (1000/100 rerun)  : {len(chunks_2)}")
print(f"Questions asked          : {len(QUESTIONS)}")
print(f"Retrieval Success Rate   : {success}/{total} = {retrieval_success_rate:.0%}")
print(f"Failures documented      : {len(failures_df)}")
print()
print(eval_df.to_string(index=False))

PART 2 SUMMARY
Documents loaded         : 10
Chunks (500/50)          : 601
Chunks (1000/100 rerun)  : 330
Questions asked          : 5
Retrieval Success Rate   : 4/5 = 80%
Failures documented      : 2

                                                             question       ground_truth_movie                                                                  ground_truth_passage top3_contains_answer rank_of_first_relevant_chunk
What crime is Andy Dufresne convicted of in The Shawshank Redemption? The Shawshank Redemption            Andy Dufresne, a banker, is convicted of murdering his wife and her lover.                  Yes                            1
       What is the name of the spinning top object used in Inception?         Inception (film)               Cobb carries a spinning top, his totem, to test whether he is dreaming.                  Yes                            1
                                          Who directed The Godfather?            The Godfather          

In [21]:
with open("results/part2_summary.md", "w") as f:
    f.write("# Part 2 summary\n\n")
    f.write(f"- Documents loaded: {len(docs)}\n")
    f.write(f"- Chunks (500/50): {len(chunks)}\n")
    f.write(f"- Chunks (1000/100 rerun): {len(chunks_2)}\n")
    f.write(f"- Retrieval Success Rate: {success}/{total} = {retrieval_success_rate:.0%}\n\n")
    f.write("## Retrieval evaluation\n\n")
    f.write(eval_df.to_markdown(index=False))
    f.write("\n\n## Chunking comparison (500/50 vs 1000/100)\n\n")
    f.write(compare_df.to_markdown(index=False))
    f.write("\n\n## Failures\n\n")
    f.write(failures_df.to_markdown(index=False))

print(open("results/part2_summary.md").read())

# Part 2 summary

- Documents loaded: 10
- Chunks (500/50): 601
- Chunks (1000/100 rerun): 330
- Retrieval Success Rate: 4/5 = 80%

## Retrieval evaluation

| question                                                              | ground_truth_movie       | ground_truth_passage                                                                  | top3_contains_answer   | rank_of_first_relevant_chunk   |
|:----------------------------------------------------------------------|:-------------------------|:--------------------------------------------------------------------------------------|:-----------------------|:-------------------------------|
| What crime is Andy Dufresne convicted of in The Shawshank Redemption? | The Shawshank Redemption | Andy Dufresne, a banker, is convicted of murdering his wife and her lover.            | Yes                    | 1                              |
| What is the name of the spinning top object used in Inception?        | Inception (film)         | C